# Ray 2.60: A Practical Introduction to LLM Post-Training

This notebook introduces supervised fine-tuning (SFT) with Ray Train, including distributed execution, data handling, model checkpoints, and the limits of recovery. It also explains how to extend the workflow to LoRA/QLoRA, DPO, and GRPO.

**Environment:** Linux with an NVIDIA GPU. A single GPU with 24 GB or more of VRAM is a conservative starting point, not a measured minimum. Multi-GPU training requires NCCL. The example uses `Qwen/Qwen2.5-0.5B-Instruct`; check model access, licensing, and memory requirements. Use Linux or WSL2 for GPU training rather than native Windows.

> Version check: November 17, 2026. Ray's official documentation and PyPI reported version 2.60.0. Recheck the official API when upgrading.

## 1. Post-Training and the Ray Components

SFT optimizes next-token loss using instruction/answer examples. DPO directly optimizes a policy from preference pairs. GRPO/PPO also require rollouts and rewards from a model or a rule-based function. Ray provides distributed scheduling: Ray Data handles data pipelines, Ray Train coordinates training workers, Ray Tune supports hyperparameter search, and Ray Serve can host online inference.

Start with the inspectable SFT workflow in this notebook. Larger preference-optimization systems often use frameworks such as verl, TRL, or OpenRLHF, with Ray coordinating applicable data, training, or rollout workloads.

In [ ]:
# First install a CUDA-compatible PyTorch build: https://pytorch.org/get-started/locally/
# These selected dependency versions have not been GPU integration-tested here; review upgrades carefully.
%pip install "ray[train,data,default]==2.60.0" "transformers==4.57.1" "datasets==4.2.0" "accelerate==1.10.1" "peft==0.17.1" safetensors
# Restart the kernel after installation; record the successful environment with pip freeze.

In [ ]:
import os
import ray
import torch

print("Ray:", ray.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Start Ray and Inspect Resources

Use `ray.init()` for local experiments. For multi-node training, connect to an existing Ray cluster with matching Ray versions, Python environments, and dependencies across the head and workers. Runtime environments can install worker dependencies; larger deployments commonly use a consistent container image.

In [ ]:
if not ray.is_initialized():
    ray.init()  # For an existing cluster, use ray.init(address="auto")

print(ray.cluster_resources())
print(ray.available_resources())

## 2.1 Learn Tasks, ObjectRefs, and Actors on CPU

A task handles independent computation, such as scoring an answer with a reward rule. An actor is a stateful process, such as a rollout model kept resident on a GPU.
`.remote()` returns an ObjectRef immediately; `ray.get()` waits for the result. Avoid calling `get` after each submission inside a loop, as that serializes the waiting.
The following example computes toy rewards. It does not call an LLM or implement RL training.

In [ ]:
@ray.remote(num_cpus=1)
def rule_reward(answer, expected):
    return float(answer.strip() == expected)

refs = [rule_reward.remote(answer, "4") for answer in ["4", "5", " 4 "]]
print("parallel rewards:", ray.get(refs))

@ray.remote(num_cpus=1)
class RewardCounter:
    def __init__(self):
        self.count = 0
    def record(self, rewards):
        self.count += len(rewards)
        return {"samples": self.count, "mean_reward": sum(rewards) / len(rewards)}

counter = RewardCounter.remote()
print(ray.get(counter.record.remote(ray.get(refs))))
ray.kill(counter)  # Release the actor's CPU resources

**Resource requests are not memory limits.** `num_gpus=1` is a scheduling declaration; it does not cap VRAM. Fractional GPU allocations do not isolate GPU memory either.
Load a GPU actor's model once in `__init__` and reuse it for inference. Training and rollout processes that occupy all GPUs can leave other workers waiting or cause resource deadlocks.
At scale, use `ray.wait()` to bound outstanding ObjectRefs instead of accumulating every generated result in the object store.

## 3. Build a Small Instruction Dataset

Use a few synthetic examples to check the code path, not to measure model quality. For production, use authorized, deduplicated, cleaned data with sensitive information handled appropriately. Separate training and validation examples, and inspect formatting, length distributions, duplication, and private information. The chat template must match the model.

In [ ]:
from datasets import Dataset

# Split by distinct question before repeating training samples to prevent validation leakage.
examples = [
    {"instruction": "Explain Ray in one sentence.", "response": "Ray is a framework for distributed Python computing."},
    {"instruction": "What is SFT?", "response": "SFT continues model training using examples with target answers."},
    {"instruction": "How do you reverse a list in Python?", "response": "You can use items[::-1]."},
    {"instruction": "What is a checkpoint used for?", "response": "It saves model and training state for evaluation or recovery."},
]
train_ds = {
    "train": Dataset.from_list(examples[:3] * 32),
    "test": Dataset.from_list(examples[3:]),
}
train_ds

## 4. Preprocess with Ray Data

Ray Data parallelizes file reading, filtering, transformation, and partitioning. Here, we convert Hugging Face Datasets into Ray Datasets and apply the model's chat template. Tokenization happens inside each training worker. At scale, decide whether to tokenize ahead of time based on throughput, CPU/GPU utilization, and caching requirements.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# Pin a model commit hash for reproducible experiments instead of the moving main branch.
MODEL_REVISION = "main"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

ray_train_ds = ray.data.from_huggingface(train_ds["train"]).map(to_chat_text)
ray_valid_ds = ray.data.from_huggingface(train_ds["test"]).map(to_chat_text)
ray_train_ds.take(2)

## 5. Ray Train + LoRA SFT

LoRA is enabled by default: only adapters on selected attention projections are trained. Set `USE_LORA=False` for full fine-tuning.
Ray starts one process per GPU. Transformers/Accelerate establishes DDP, and Trainer applies distributed sampling to the same HF Dataset on each worker.
**Do not also shard this dataset manually with Ray: that would shard it twice.**

This small example uses `take_all()` to collect processed rows. Do not copy this approach for large datasets. Instead, use Ray Train `datasets` / `get_dataset_shard()` with an appropriate custom DataLoader, or let every worker read the same disk-backed HF Dataset and use Trainer's distributed sampling.

For simplicity, the loss covers the entire chat sequence, including user tokens. Instruction SFT often supervises only assistant tokens. Generate label masks using the model's template and inspect them sample by sample; do not infer token boundaries from character positions alone.

Effective global batch size = per-device batch size × gradient accumulation steps × worker count. With DDP, each GPU still holds the complete base model. LoRA does not shard model weights.

In [ ]:
import tempfile
from pathlib import Path
from ray import train
from ray.train import Checkpoint, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from ray.train.huggingface.transformers import prepare_trainer

# Collect rows on the driver only for this tiny demonstration.
train_rows = ray_train_ds.take_all()
eval_rows = ray_valid_ds.take_all()
MAX_LENGTH = 256
assert train_rows and eval_rows
print("train/eval rows:", len(train_rows), len(eval_rows))

In [ ]:
def train_loop_per_worker(config):
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
        DataCollatorForSeq2Seq, set_seed,
    )
    from peft import LoraConfig, get_peft_model

    set_seed(42)
    context = train.get_context()
    tok = AutoTokenizer.from_pretrained(config["model_id"], revision=config["revision"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        config["model_id"], revision=config["revision"],
        torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
    )
    model.config.use_cache = False
    if config["use_lora"]:
        model = get_peft_model(model, LoraConfig(
            task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
        ))
        model.print_trainable_parameters()

    def tokenize(batch):
        encoded = tok(batch["text"], truncation=True,
                      max_length=config["max_length"], add_special_tokens=False)
        encoded["labels"] = [ids.copy() for ids in encoded["input_ids"]]
        return encoded

    def make_dataset(rows):
        return Dataset.from_list(rows).map(tokenize, batched=True, remove_columns=["text"])

    # Mask padding labels with -100 while preserving supervision on real EOS tokens.
    collator = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, pad_to_multiple_of=8)
    with tempfile.TemporaryDirectory(prefix="ray-sft-") as workdir:
        args = TrainingArguments(
            output_dir=workdir, max_steps=config["max_steps"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=config["grad_accum"],
            learning_rate=config["learning_rate"],
            logging_steps=1, save_strategy="no", eval_strategy="no",
            bf16=use_bf16, fp16=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            ddp_find_unused_parameters=False, report_to=[], seed=42,
        )
        hf_trainer = Trainer(
            model=model, args=args, data_collator=collator,
            train_dataset=make_dataset(config["train_rows"]),
            eval_dataset=make_dataset(config["eval_rows"]),
        )
        hf_trainer = prepare_trainer(hf_trainer)
        output = hf_trainer.train()
        metrics = hf_trainer.evaluate()
        metrics["train_loss"] = output.training_loss
        checkpoint = None
        if context.get_world_rank() == 0:
            export = Path(workdir) / "export"
            hf_trainer.save_model(str(export))
            tok.save_pretrained(str(export))
            checkpoint = Checkpoint.from_directory(str(export))
        # Every rank calls report equally often; only rank 0 uploads model files.
        train.report(metrics, checkpoint=checkpoint)

## 6. Launch Training

This smoke run performs 10 optimizer steps to check the workflow. It does not establish an improvement in model quality.
The first run downloads the model. Set `RUN_TRAINING=True` below to launch training.

The default LoRA learning rate is 2e-4; choose a separate rate for full fine-tuning, for example starting experiments at 2e-5.
A local `storage_path` is suitable only for a single node. Multi-node runs need persistent storage accessible from all nodes.
Automatic retries are not enabled here, so a persistent GPU OOM does not repeatedly restart an unsuccessful job.

In [ ]:
RUN_TRAINING = False
NUM_WORKERS = 1
USE_LORA = True
result = None
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Run in a CUDA-enabled Linux or WSL2 environment."
    assert ray.cluster_resources().get("GPU", 0) >= NUM_WORKERS
    trainer = TorchTrainer(
        train_loop_per_worker,
        train_loop_config={
            "model_id": MODEL_ID, "revision": MODEL_REVISION,
            "max_steps": 10, "max_length": MAX_LENGTH,
            "batch_size": 1, "grad_accum": 4,
            "learning_rate": 2e-4 if USE_LORA else 2e-5,
            "use_lora": USE_LORA,
            "train_rows": train_rows, "eval_rows": eval_rows,
        },
        scaling_config=ScalingConfig(
            num_workers=NUM_WORKERS, use_gpu=True,
            resources_per_worker={"CPU": 1, "GPU": 1},
        ),
        run_config=RunConfig(
            name="ray-llm-sft", storage_path=str(Path("ray_results").resolve()),
        ),
    )
    result = trainer.fit()
    print(result.metrics)
    print(result.checkpoint)
else:
    print("GPU training skipped. Set RUN_TRAINING=True when the environment is ready.")

## 7. Load the Adapter or Full Model for Generation

A LoRA checkpoint contains the adapter and tokenizer; inference also needs the original base model. This example generates on CPU, which can be slow. Move the model and inputs to an available CUDA device for faster inference.
The validation set contains only one independent question and is not a credible benchmark. For real evaluation, fix the task set, model revision, and decoding parameters, then compare the base and fine-tuned models for correctness, format compliance, and safety.

**Recovery boundary:** This example exports model/adapter weights without optimizer, scheduler, or RNG state. It cannot resume training exactly. Exact recovery requires a complete HF Trainer checkpoint, transported through a Ray checkpoint and loaded inside each worker using `trainer.train(resume_from_checkpoint=...)`. All ranks must resume from the same training step.

In [ ]:
if result is not None:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    with result.checkpoint.as_directory() as checkpoint_dir:
        tok = AutoTokenizer.from_pretrained(checkpoint_dir)
        if USE_LORA:
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float32)
            model = PeftModel.from_pretrained(base, checkpoint_dir)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                checkpoint_dir, torch_dtype=torch.float32)
        model.eval()
        messages = [{"role": "user", "content": "What is a checkpoint used for?"}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
else:
    print("Run this cell again after training.")

## 8. Scale from One Machine to a Cluster

- Start Ray head and worker nodes. Confirm that `ray status` reports the expected GPUs, then connect with `ray.init(address="auto")`.
- `ScalingConfig(num_workers=N, use_gpu=True)` sets the number of training processes. DDP, FSDP, and DeepSpeed differ in memory partitioning and communication. Models that exceed per-device memory may require FSDP or DeepSpeed.
- Point `RunConfig(storage_path=...)` to persistent storage accessible to all nodes. Verify recovery behavior, checkpoint completeness, and dependency consistency.
- Use Ray Data to read partitioned Parquet/JSONL files, filter invalid examples, manage sequence lengths, and monitor throughput. Avoid loading the entire dataset into the notebook driver.
- Measure tokens/s, GPU utilization, data wait time, and communication overhead. More GPUs do not guarantee linear speedups.

## 9. Next Steps: LoRA, DPO, and GRPO

**LoRA/QLoRA:** PEFT injects adapters after model loading and trains only adapter parameters, reducing optimizer-state and gradient memory. QLoRA also needs a compatible bitsandbytes environment for 4-bit weights. Verify CUDA, bitsandbytes, and PEFT compatibility on every worker.

**DPO:** Prepare `(prompt, chosen, rejected)` preference pairs and use TRL's `DPOTrainer` or another suitable training loop. Ray Train manages workers and resources. Pay attention to reference-model memory, sequence length, and preference quality.

**GRPO/PPO:** Schedule rollout inference and reward computation alongside policy training. Resource orchestration, asynchronous sampling, and checkpoint behavior become more complex. A dedicated framework such as verl provides a more suitable starting point; follow its own Ray deployment instructions.

Production workflows also need versioned data, recoverable persistent checkpoints, experiment tracking, offline evaluations, model cards and license checks, safety evaluations, and training/inference format compatibility checks.

## Official References

- [Ray Train](https://docs.ray.io/en/latest/train/train.html)
- [Ray Train + Hugging Face Transformers](https://docs.ray.io/en/latest/train/api/doc/ray.train.huggingface.transformers.prepare_trainer.html)
- [Ray Data](https://docs.ray.io/en/latest/data/data.html)
- [Ray examples](https://docs.ray.io/en/latest/ray-overview/examples.html)
- [Ray on GitHub](https://github.com/ray-project/ray)

This notebook demonstrates basic Ray training orchestration. Ray APIs, Transformers Trainer arguments, and integration support can change across versions. Consult the corresponding documentation before upgrading.

## 10. DPO and GRPO: Data and Resource Requirements

| Method | Data | Main model roles | First checks |
|---|---|---|---|
| SFT | Prompt + answer | Policy | Assistant label mask, template |
| DPO | Prompt + chosen + rejected | Policy, reference | Pair quality, policy/reference initialization |
| GRPO | Prompt, verifiable answer or reward function | Policy, rollout, optional reference depending on configuration | Within-group reward variation, KL, length bias |

DPO optimizes relative log probabilities of preferred and rejected answers to the same prompt; it does not require online rollouts.
GRPO samples a group of answers per prompt, constructs advantages from relative rewards, and updates the policy.
When every reward in a group is identical, normalization provides no useful relative reward signal. Track within-group reward standard deviation, KL, output length, and external evaluation results in addition to mean reward.

Ray orchestrates processes and resources; it does not automatically implement the correct DPO/GRPO loss or weight synchronization.
TRL is a useful starting point for single-machine algorithm experiments. verl and OpenRLHF have their own Ray orchestration and version requirements. Use their prescribed environments rather than assuming compatibility with this notebook's dependencies.

In [ ]:
# Minimal preference-data structure; this does not launch training.
preference_pair = {
    "prompt": [{"role": "user", "content": "What is 2+2?"}],
    "chosen": [{"role": "assistant", "content": "4"}],
    "rejected": [{"role": "assistant", "content": "5"}],
}
assert preference_pair["chosen"] != preference_pair["rejected"]

# Illustrative GRPO advantages; real training also needs policy loss, KL, clipping, and more.
rewards = torch.tensor([1., 0., 1., 0.])
advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-6)
print("group advantages:", advantages.tolist())

## 11. Troubleshooting and Exercises

| Symptom | What to inspect |
|---|---|
| Workers remain PENDING | `ray.available_resources()`; actors holding CPU/GPU resources |
| CUDA OOM | Reduce length/batch size; use LoRA; DDP does not shard base weights |
| Multi-node NCCL hangs | Network connectivity, CUDA/drivers, matching communication steps across ranks |
| NaN loss | Labels, learning rate, precision, hardware support for bf16 |
| Checkpoint cannot resume training | Adapter export versus complete optimizer/RNG training state |
| Driver slows down as data grows | Replace `take_all()` with partitioned storage and streaming |

Exercises:

1. Use two GPUs and calculate the effective global batch size.
2. Compare throughput while keeping global batch size fixed.
3. Print tokens and label masks for three samples.
4. Replace the toy validation data with an independent evaluation set.
5. Add complete training checkpoints, interrupt a run, and verify the resumed step.

GPU training has not been executed for this notebook. No measured loss, throughput, or quality improvement is claimed. Syntax validation does not replace an integration test on the target hardware.

### Additional Official Links

- [Ray 2.60.0 release source](https://github.com/ray-project/ray/tree/ray-2.60.0)
- [Ray Core concepts](https://docs.ray.io/en/latest/ray-core/key-concepts.html)
- [Ray Hugging Face integration source](https://github.com/ray-project/ray/blob/ray-2.60.0/python/ray/train/huggingface/transformers/_transformers_utils.py)
- [TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)
- [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

"Latest" refers to the Ray release checked on the stated date. This tutorial does not claim to use the newest model or the newest version of every dependency.

## 12. Where vLLM and SGLang Fit

Inference engines are important in post-training even when they do not compute gradients. They generate synthetic SFT data, run evaluation, and produce online rollouts for algorithms such as GRPO/PPO. Offline DPO uses fixed preference pairs and does not require an online inference engine during optimization.

| Layer | Responsibility | Examples |
|---|---|---|
| Orchestration | Schedule processes and reserve CPU/GPU resources | Ray |
| Gradient computation | Forward/backward passes and optimizer updates | Transformers, PyTorch, FSDP, DeepSpeed |
| Generation | Batch requests, manage KV cache, sample output tokens | vLLM, SGLang |
| Algorithm integration | Rewards, policy loss, rollout collection, weight synchronization | verl, OpenRLHF, TRL integrations |

vLLM and SGLang can run without Ray. An engine's tensor-parallel execution is different from Ray scheduling independent engine replicas. Do not assume a standalone HTTP server is managed by Ray merely because both run on the same machine.

The following sections are optional and can be studied without running SFT. They use the base model, not the LoRA adapter exported earlier. Loading an adapter or a merged checkpoint requires an explicit backend-supported workflow.

## 13. Inference Concepts That Affect Rollout Throughput

| Concept | Meaning | Post-training implication |
|---|---|---|
| Prefill | Process input tokens and build attention KV state | Long prompts can dominate time to first token |
| Decode | Generate successive output tokens using cached state | Long answers increase rollout time and KV memory |
| KV cache | Per-layer key/value tensors retained for attention | Weights fitting in VRAM does not guarantee large batches fit |
| Continuous batching | Admit and retire requests as generation progresses | Helps heterogeneous request lengths share the GPU |
| Prefix caching | Reuse eligible KV state for identical token prefixes | Repeated system prompts may reduce prefill work; measure cold and warm cache separately |
| Tensor parallelism (TP) | Partition model computation across GPUs | Fits larger models but adds communication; TP is not independent replicas |

Time to first token (TTFT) measures how quickly an answer begins. Inter-token latency measures decode responsiveness. Aggregate output tokens/s measures throughput across requests. A configuration optimized for maximum throughput may have worse tail latency.

KV memory grows roughly with active sequence tokens × layers × KV heads × head dimension × two tensors (K and V) × bytes per element, with backend-specific partitioning and overhead. Model weight memory alone is not a capacity estimate.

Both engines support high-throughput serving and evolving caching/scheduling optimizations. vLLM is associated with paged KV-cache management; SGLang emphasizes prefix reuse through RadixAttention and related runtime optimizations. These features are not exclusive capability boundaries. Benchmark your exact model, backend version, hardware, and request distribution.

## 14. Start One Backend in Its Own Environment

Use a **separate Linux shell and environment** for each engine. Do not install both engines over the pinned SFT environment: their PyTorch, CUDA, kernel, and Python requirements may differ. Follow the current installation guide, select a supported release, and record `pip freeze`, model revision, GPU, driver, and CUDA versions. These commands are opt-in shell instructions, not automatically executed notebook cells.

### Option A: vLLM

The official quickstart currently recommends `uv` with automatic PyTorch backend selection. Activate a dedicated environment first:

```bash
python -m venv .venv-vllm
source .venv-vllm/bin/activate
python -m pip install uv
uv pip install vllm --torch-backend=auto
vllm serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000
```

### Option B: SGLang

Create a different environment, install the compatible SGLang release using its [installation guide](https://docs.sglang.io/), then launch:

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8001
```

Use one server at a time on a single GPU. Stop it with Ctrl+C before launching the other, and confirm its GPU processes have exited. Wait for the server's ready message before querying it. Do not run these servers alongside the earlier training job without a deliberate memory and device allocation.

For larger models, vLLM exposes `--tensor-parallel-size`; SGLang exposes `--tp-size`. Inspect the installed release's `--help` and reserve the corresponding number of GPUs. Setting TP does not create extra hardware.

The examples bind to loopback and assume no local server authentication. If the notebook runs on another host, loopback refers to that host; use an appropriate reachable endpoint or tunnel. Configure authentication before exposing a service beyond your local development environment.

In [ ]:
# Install only the lightweight client in this notebook's kernel if needed.
# %pip install openai

RUN_INFERENCE = False
BACKEND = "vllm"  # Change to "sglang" after starting that server.
ENDPOINTS = {
    "vllm": "http://127.0.0.1:8000/v1",
    "sglang": "http://127.0.0.1:8001/v1",
}
INFERENCE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def make_local_client(backend):
    import os
    from openai import OpenAI
    return OpenAI(
        base_url=ENDPOINTS[backend],
        api_key=os.environ.get("LOCAL_INFERENCE_API_KEY", "EMPTY"),
        timeout=120.0, max_retries=0,
    )

if RUN_INFERENCE:
    with make_local_client(BACKEND) as client:
        available_models = [m.id for m in client.models.list().data]
        print("Served model IDs:", available_models)
        assert INFERENCE_MODEL in available_models, "Use the exact served model ID."
        response = client.chat.completions.create(
            model=INFERENCE_MODEL,
            messages=[{"role": "user", "content": "Explain KV caching in two sentences."}],
            temperature=0.0, top_p=1.0, max_tokens=96,
        )
        print(response.choices[0].message.content)
        print("Usage:", response.usage)
else:
    print("Start a backend, then set RUN_INFERENCE=True to send requests.")

### 14.1 Offline vLLM Generation

For offline dataset generation, vLLM's Python API avoids a separately managed HTTP server. Run the next cell only in a vLLM-compatible kernel with available GPU memory. It is an alternative to the server example, not an additional process to run on the same occupied GPU.

`LLM.chat()` accepts chat messages and applies the model's chat template. With `LLM.generate()` and raw prompts, apply the template explicitly. Do not template twice. The engine can retain GPU allocations after a call; restart the dedicated kernel when finished.

In [ ]:
RUN_OFFLINE_VLLM = False
if RUN_OFFLINE_VLLM:
    from vllm import LLM, SamplingParams
    engine = LLM(model=INFERENCE_MODEL, tensor_parallel_size=1)
    conversations = [
        [{"role": "user", "content": "What is supervised fine-tuning?"}],
        [{"role": "user", "content": "What is a rollout in reinforcement learning?"}],
    ]
    samples = engine.chat(
        conversations,
        sampling_params=SamplingParams(temperature=0.7, top_p=0.9, max_tokens=64),
    )
    for sample in samples:
        print(sample.outputs[0].text)
else:
    print("Offline vLLM generation is disabled.")

## 15. Measure Both Backends Under the Same Workload

Run the next cell once per backend, changing `BACKEND` and switching the server between runs. Keep GPU count, model revision, dtype/quantization, prompt set, output cap, concurrency, and cache state comparable. Explicit sampling arguments reduce hidden differences, but backend defaults and kernels can still differ.

This small non-streaming probe reports end-to-end request latency and aggregate output tokens/s. It includes client/network overhead and is not a rigorous serving benchmark. It does not measure TTFT or inter-token latency; those need streaming timestamps. Repeating a prompt may warm prefix caches, so record that condition. A faster run is not necessarily more accurate.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

RUN_BENCHMARK = False
benchmark_results = globals().get("benchmark_results", {})
benchmark_prompts = [
    "Explain gradient accumulation in two sentences.",
    "Explain tensor parallelism in two sentences.",
    "Explain the KV cache in two sentences.",
    "Explain a validation dataset in two sentences.",
] * 2

def benchmark_request(prompt):
    with make_local_client(BACKEND) as client:
        start = perf_counter()
        response = client.chat.completions.create(
            model=INFERENCE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0, top_p=1.0, max_tokens=64,
        )
        elapsed = perf_counter() - start
    if response.usage is None or response.usage.completion_tokens is None:
        raise ValueError("The backend did not report completion token usage.")
    return elapsed, response.usage.completion_tokens

if RUN_BENCHMARK:
    start = perf_counter()
    with ThreadPoolExecutor(max_workers=2) as pool:
        measurements = list(pool.map(benchmark_request, benchmark_prompts))
    wall = perf_counter() - start
    durations = sorted(item[0] for item in measurements)
    benchmark_results[BACKEND] = {
        "requests": len(measurements), "concurrency": 2,
        "wall_seconds": wall,
        "mean_request_seconds": sum(durations) / len(durations),
        "max_request_seconds": max(durations),
        "output_tokens_per_second": sum(item[1] for item in measurements) / wall,
    }
    print(benchmark_results)
else:
    print("Benchmark disabled; no performance numbers have been measured.")

## 16. Use Ray to Orchestrate Rollout Requests

The example below sends independent generation requests from Ray tasks to an existing engine server, then scores the answers with an exact-match rule. These HTTP client tasks reserve CPU resources only. **They do not reserve the server's GPU.** The externally launched server owns that GPU independently of Ray.

This example is single-node: workers use the same loopback endpoint as the notebook. On a cluster, use a reachable service address and ensure `openai` is installed in worker environments. For simple HTTP concurrency, the previous thread pool is sufficient; Ray becomes useful when the surrounding workflow also distributes scoring, preprocessing, or other compute.

For a Ray-managed GPU engine, a long-lived actor should reserve GPUs and initialize its engine once. A TP engine needs a coordinated reservation of all required GPUs. Combining an outer actor reservation with an engine that requests another set of Ray GPU workers can deadlock. Use the integration framework's supported placement strategy instead of inventing nested reservations.

In [ ]:
RUN_RAY_ROLLOUT = False
rollout_records = []

if RUN_RAY_ROLLOUT:
    import os
    import ray
    if not ray.is_initialized():
        ray.init()

    @ray.remote(num_cpus=1, max_retries=0)
    def generate_rollout(base_url, model_id, prompt_id, sample_id, prompt):
        import os
        from openai import OpenAI
        with OpenAI(
            base_url=base_url,
            api_key=os.environ.get("LOCAL_INFERENCE_API_KEY", "EMPTY"),
            timeout=120.0, max_retries=0,
        ) as client:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8, top_p=0.95, max_tokens=32,
            )
        answer = response.choices[0].message.content or ""
        return {
            "prompt_id": prompt_id, "sample_id": sample_id,
            "prompt": prompt, "answer": answer,
            "finish_reason": response.choices[0].finish_reason,
        }

    tasks = [("addition", i, "What is 2+2? Reply with the integer only.") for i in range(4)]
    pending = []
    for prompt_id, sample_id, prompt in tasks:
        pending.append(generate_rollout.remote(
            ENDPOINTS[BACKEND], INFERENCE_MODEL, prompt_id, sample_id, prompt))
        if len(pending) >= 2:  # Bound outstanding requests.
            ready, pending = ray.wait(pending, num_returns=1)
            rollout_records.extend(ray.get(ready))
    rollout_records.extend(ray.get(pending))
    rollout_records.sort(key=lambda row: (row["prompt_id"], row["sample_id"]))
    for row in rollout_records:
        row["reward"] = float(row["answer"].strip() == "4")
    print(rollout_records)
else:
    print("Ray rollout requests are disabled.")

### 16.1 Group Rewards Are Only One Part of GRPO

The exact-match reward is intentionally simple: `4.` and an explanation containing `4` receive zero. Real verifiers must define normalization and correctness carefully. Keep samples grouped by prompt; never normalize across unrelated questions. The next cell computes a population-standard-deviation normalization for illustration. Implementations may use different normalization conventions.

Neither this cell nor the HTTP example computes a policy gradient. No policy update or weight synchronization occurs.

In [ ]:
from collections import defaultdict
import math

groups = defaultdict(list)
for row in rollout_records:
    groups[row["prompt_id"]].append(row)
for prompt_id, group in groups.items():
    assert len(group) >= 2, "GRPO requires multiple samples per prompt."
    mean_reward = sum(row["reward"] for row in group) / len(group)
    std_reward = math.sqrt(sum((row["reward"] - mean_reward) ** 2 for row in group) / len(group))
    for row in group:
        row["illustrative_advantage"] = (row["reward"] - mean_reward) / (std_reward + 1e-6)
    print(prompt_id, {"mean_reward": mean_reward, "std_reward": std_reward})
    if std_reward == 0:
        print("No within-group relative reward signal for this prompt.")

## 17. From Generation to a Correct Training Loop

The synchronous integration pattern is:

```text
publish policy version k to rollout workers
    -> generate groups of answers tagged with version k
    -> verify rewards and prepare token-level training data
    -> compute policy loss and update trainable weights
    -> synchronize all rollout workers to version k+1
    -> invalidate or refresh stale caches as required by the backend
    -> start the next rollout group
```

| Required item | Why it matters |
|---|---|
| Exact prompt/completion token IDs and masks | Retokenizing displayed text can change token boundaries; distinguish padding, EOS, and truncation |
| Behavior-policy version and log probabilities | PPO/GRPO variants need correctly aligned likelihoods for their policy-ratio calculation |
| Sampling configuration | Temperature, top-p/top-k, repetition penalties, and stopping rules affect the behavior distribution |
| Reference-policy outputs, when used | Compute the configured KL penalty against the intended reference |
| Group identity and reward provenance | Keep answers associated with their prompt, verifier version, and policy version |
| Synchronized weights or adapters | A training update does not automatically update an HTTP inference server |

Some OpenAI-compatible endpoints expose `logprobs`, but support and semantics vary by backend, model, and release. Returned text/token strings are not necessarily an exact training-ready token-ID stream. Sampling transformations and serving/training numerical differences also need attention. A framework may recompute behavior-policy log probabilities or apply correction mechanisms; follow that implementation's contract.

Do not label a rollout with a policy version unless the server has actually acknowledged loading that version. A synchronous workflow waits for all rollout replicas to finish updating before accepting new work. An asynchronous workflow must track and bound policy lag and handle stale samples explicitly.

Choose a framework-supported weight-transfer path: full checkpoints, adapter loading, or in-memory distributed transfer are different mechanisms. Confirm whether synchronization also handles tokenizer/template consistency and invalidation of KV/prefix caches created under old weights. Exporting a checkpoint from Ray Train does not perform any of these steps automatically.

### 17.1 GPU Placement and Framework Selection

| Layout | Advantage | Main constraint |
|---|---|---|
| Separate training and rollout GPUs | Training and generation can overlap | Additional VRAM/devices and weight-transfer traffic |
| Shared GPUs in alternating phases | Reuse scarce GPUs | Explicit release, sleep/offload, or framework-managed memory transitions |
| Multiple inference replicas | More independent rollout capacity | Route requests, synchronize versions, and bound queues |
| Tensor-parallel engine | Partition a larger model across GPUs | Coordinated reservation and high-bandwidth communication |

On a single GPU, begin with sequential experiments: stop training workers before launching serving, and stop the server before resuming training. GPU fractions do not provide memory isolation. Ray placement groups can coordinate resource reservations, but do not automatically make training and serving memory compatible.

For actual GRPO/PPO work, select a documented verl or OpenRLHF backend integration and follow its pinned environment. TRL also offers specific vLLM integrations for supported trainers and modes. Do not assume arbitrary versions of Ray, vLLM, SGLang, and the training framework are interchangeable. SGLang and vLLM support varies by framework release and model architecture.

Evaluate: supported model/adapter formats, token/log-prob semantics, weight-transfer method, cache invalidation, rollout/training co-location, version tracking, and recovery from a failed worker. A passing generation smoke test is only the first check.

## 18. Inference Exercises, Sources, and Validation Limits

1. Generate the same prompts with each backend and inspect format adherence and completion lengths.
2. Measure one-request latency and two-request concurrency with matching parameters; record cold and warm cache conditions.
3. Add streaming timestamps for TTFT and inter-token latency rather than inferring them from total request time.
4. Inspect a backend's token/log-prob response and compare it with training-side tokenization before using it in a policy loss.
5. In a supported RL framework, update the policy once and verify that every rollout worker acknowledges the new version.

Official sources checked on November 17, 2026:

- [vLLM quickstart](https://docs.vllm.ai/en/latest/getting_started/quickstart/)
- [vLLM official quickstart source](https://github.com/vllm-project/vllm/blob/main/docs/getting_started/quickstart.md) — used to verify `vllm serve`, `LLM`, `SamplingParams`, and chat generation because the rendered documentation returned HTTP 403 during this check.
- [SGLang request tutorial](https://docs.sglang.io/docs/basic_usage/send_request) — verifies `sglang.launch_server` and OpenAI-compatible chat requests.
- [SGLang documentation and installation](https://docs.sglang.io/)
- [TRL vLLM integration](https://huggingface.co/docs/trl/vllm_integration)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

These links track evolving documentation; they are not a tested release lock. Record the backend version you install and follow its supported hardware/software matrix. The serving and rollout examples have passed syntax and notebook-format checks only. No backend server, model download, GPU benchmark, policy update, or distributed weight synchronization has been executed as part of preparing this tutorial.

## 19. A Learning Contract for Independent Experiments

For a reader trained in research, the goal is to explain why a method works, when it fails, and how to verify it. Sections 1–18 introduce components; sections 19–27 connect correctness, experimental design, and systems diagnosis.

| Stage | Deliverable | Passing condition |
|---|---|---|
| A: Correctness | Data audit, token/label table, tiny optimization experiment | Diagnose leakage, incorrect masks, and DPO sign errors |
| B: Single GPU | Base-versus-SFT evaluation on independent data | Overfit a small set; reproduce real task scores |
| C: Scaling and recovery | Two-GPU comparison and full-checkpoint recovery log | Match global batch, avoid double sharding, restore the correct step |
| D: Preference optimization | DPO or framework GRPO comparison | Fix baselines; report KL, length, reward, and external correctness |
| E: Delivery | Performance report, environment manifest, failure analysis | Reproduce in a fresh environment and explain the main bottleneck |

Prerequisites: derive cross-entropy/log-softmax gradients and explain Adam state, DDP gradient averaging, sampling distributions, and expectations. Fill gaps with small tensor experiments. Change one experimental factor at a time.

The CPU labs below use `labs/posttrain_checks.py`. Clone the full repository and open the notebook from its root. No model downloads are required.

In [ ]:
from labs.posttrain_checks import (
    audit_splits, assistant_labels, dpo_loss, toy_dpo_update, paired_bootstrap, self_check,
)
print(self_check())

## 20. Data Contracts and Leakage Audits

Define the task, input distribution, output format, and scoring rule before collecting data. Preserve IDs, provenance, usage permissions, timestamps, task categories, versions, and cleaning records. Split by user, document, repository, or question family; paraphrases of the same problem must not be randomly spread across train and test.

The next cell checks schema, duplicate IDs, and normalized exact prompt overlap. It does not detect semantic duplication, answer leakage, benchmark contamination, or source-level near duplicates. Split before augmentation; keep test data sealed from hyperparameter selection.

Report task counts, length P50/P95/P99, truncation rate, zero-supervision examples, source proportions, and duplication rate. Truncation can change the task distribution when long answers are disproportionately removed.

In [ ]:
splits = {
    "train": [{"id": "train-1", "prompt": "What is 2+2?", "answer": "4"}],
    "test": [{"id": "test-1", "prompt": " what IS 2+2? ", "answer": "4"}],
}
issues = audit_splits(splits)
print(issues)
assert any(issue["kind"] == "cross_split_prompt" for issue in issues)
# Deliberately contaminated fixture: quarantine/re-split its source group before training.

## 21. SFT Correctness Matters More Than Falling Loss

Causal LMs usually shift logits and labels internally: logits at t predict the label at t+1. Do not shift again if the model already does so. `-100` ignores a target; it is not a vocabulary token. Supervised positions must survive the shift.

Assistant-only SFT masks user, system, and padding labels while preserving target assistant content and template-defined termination tokens. Even when EOS and PAD share an ID, distinguish real EOS from padding using positions and the attention mask.

Section 5 uses full-sequence loss as an explicit simplification. The next lab verifies mask application; it does not automatically fix a chat template. Production integration replaces worker tokenization: obtain a reliable assistant mask, truncate/pad all aligned arrays consistently, then construct labels.

Some templates support `apply_chat_template(..., return_dict=True, return_assistant_tokens_mask=True)`, but require generation markers. Do not assume the current Qwen template supports this. Use a validated template or verified token-level boundaries, and fail rather than silently reverting to full-sequence supervision. Inspect token roles, IDs, decoded text, and labels for multi-turn chats, tool calls, EOS, truncation, and empty answers.

Packing can allow cross-example attention. Decide whether isolated attention boundaries and position resets are needed, and verify kernel support. Token-averaged and sequence-averaged losses differ for variable lengths; verify normalization across accumulation steps and ranks.

In [ ]:
ids = [10, 11, 20, 2, 2]
attention = [1, 1, 1, 1, 0]
assistant = [0, 0, 1, 1, 0]
labels = assistant_labels(ids, attention, assistant)
print(list(zip(ids, attention, assistant, labels)))
print("Targets predicted by logits at positions 0..3:", labels[1:])
assert labels[-2:] == [2, -100]
# Synthetic token IDs test alignment; they do not represent a real tokenizer template.

## 22. Turn the DPO Objective into a Checkable Update

Let (x) be a prompt and (y_w,y_l) the preferred/rejected answers. The reference policy is fixed. Sequence log probabilities sum over completion tokens only:

\[
L=-\log\sigma\left(\beta\left[\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_w|x)+\log\pi_{ref}(y_l|x)\right]\right).
\]

Use log-softmax and sums rather than multiplying probabilities. Replacing sums with means changes the objective. Freeze the reference in evaluation mode and record its initialization relative to the policy, especially when continuing from SFT.

The next lab uses a two-answer policy parameterized by one log-odds scalar and a uniform reference. It performs actual DPO gradient updates and checks the gradient with finite differences, but does not fine-tune an LLM. Initial policy/reference equality gives loss log(2). Always preferring the chosen answer increases its probability; reversing preferences should reverse the direction.

For real DPO, verify preference quality, shared prompts, truncation, reference log probabilities, and answer-length distributions. A growing preference margin is not evidence of better task correctness.

In [ ]:
import math
trace = toy_dpo_update()
print(trace[0], trace[-1])
assert math.isclose(dpo_loss(-2, -3, -2, -3), math.log(2))
assert trace[-1]["chosen_probability"] > trace[0]["chosen_probability"]
assert trace[-1]["loss"] < trace[0]["loss"]

## 23. Close the Evaluation Loop

Fix independent prompts, evaluator version, decoding settings, and task slices before comparing base → SFT → DPO/GRPO. Save per-prompt raw outputs and scores. Prefer deterministic verifiers where possible; audit model judges for ordering, verbosity, and other biases using human review.

Choose task metrics such as math accuracy, executable test pass rate, or structured-output validity. Also report length, refusal, timeout, and task-slice metrics. Execute generated code in an isolated environment. Separate training reward from independent evaluation to detect reward hacking.

The next example compares paired scores on the same prompts with a bootstrap interval. Scores are synthetic, not model measurements. Resample independent prompts or source groups; multiple outputs for one prompt are not independent questions. Small-sample intervals are unstable and cannot establish effectiveness.

Fix the sampling budget before reporting pass@k, and do not compare different k values or generation budgets as equivalent. Repeat training with multiple seeds and report more than the best run.

In [ ]:
base_scores = [0, 1, 0, 1, 0, 1, 0, 1]
candidate_scores = [1, 1, 0, 1, 1, 1, 0, 1]
print("Synthetic paired comparison:", paired_bootstrap(base_scores, candidate_scores))

## 24. Memory Budgets, Distributed Normalization, and Profiling

Training memory includes weights, gradients, optimizer states, activations, communication buffers, and fragmentation. A conventional mixed-precision full Adam setup is often estimated at roughly 16 bytes per parameter for static states, depending on implementation, before activations. LoRA reduces trainable-parameter states but retains base weights and activations.

DDP replicates model and optimizer state. ZeRO-1/2/3 progressively partitions optimizer states, gradients, and parameters. Configured FSDP sharding adds all-gather/reduce-scatter traffic. TP partitions within layers; pipeline parallelism partitions layers and introduces pipeline bubbles. Select based on memory and interconnect constraints.

| Observation | Hypothesis to check | Controlled comparison |
|---|---|---|
| Idle GPU, busy CPU | Tokenization or input stalls | Cache a small batch and compare step time |
| More GPUs are slower | Communication or tiny microbatches | Fix global batch; separate compute/communication |
| Spare memory, low throughput | Small batches or padding | Length bucketing; measure useful tokens/s |
| Long rollout tail | A few very long answers | Inspect length quantiles, finish reasons, truncation |
| Growing object store | Unconsumed ObjectRefs | Bound outstanding tasks and consume results |

Distinguish microsteps from optimizer steps. Global sample batch = microbatch × accumulation × world size; effective token counts still vary with length. For one-versus-two-GPU gradient checks, fix samples, seed, dropout, and normalization, allowing floating-point reduction differences. One logged mean loss is insufficient.

Profile a short warmed-up window with PyTorch profiler or Nsight. GPU work is asynchronous; manual timing needs appropriate synchronization. Report averages, tail latency, and profiling overhead.

## 25. Recovery, Idempotency, and Experiment Manifests

A model export is not a resumable training checkpoint. Preserve model/adapters, optimizer, scheduler, AMP scaler if used, RNG state, global step, data order/cursor, accumulation boundary, and framework configuration. Prefer checkpointing at optimizer-step boundaries to avoid losing partial accumulated gradients.

Acceptance test: run N steps with a fixed seed; separately stop at K, save, and resume to N. Compare step, learning rate, data IDs, optimizer state, and parameter differences. Nondeterministic kernels may prevent bitwise identity; define tolerances in advance. Successful loading alone does not establish correct recovery.

Ray task retries can duplicate external side effects. Use stable sample IDs with atomic writes/deduplication. Actor restarts do not automatically restore in-memory state; rollout workers must re-acknowledge policy versions. Retries do not fix persistent OOM or bad data.

The following cell writes a local manifest without inventing hardware or metrics. Model revision, data hash, environment lock, code commit, and evaluation configuration should be fixed inputs to each experiment.

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
from pathlib import Path

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

git_result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
manifest = {
    "python": platform.python_version(),
    "git_commit": git_result.stdout.strip() if git_result.returncode == 0 else None,
    "packages": {p: package_version(p) for p in ["ray", "torch", "transformers", "peft", "vllm", "sglang"]},
    "model_revision": "REPLACE_WITH_COMMIT_HASH",
    "dataset_sha256": "REPLACE_WITH_REAL_DATA_HASH",
    "seed": 42, "hardware": "RECORD_ON_TARGET_HOST",
    "evaluation_config": "REPLACE_WITH_FROZEN_CONFIG",
    "status": "manifest_template_not_a_training_result",
}
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
(artifact_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest)

## 26. GRPO Failure Modes and Experimental Design

Group normalization is one advantage construction, not a complete algorithm. Implementations also specify token/sequence objectives, importance ratios, clipping, KL, masks, and normalization denominators. Follow the selected framework version rather than mixing fragments from different variants.

| Symptom | Possible explanation | Diagnostic |
|---|---|---|
| Higher reward without better external accuracy | Reward hacking or judge bias | Review samples with a frozen external verifier |
| Identical rewards in every group | Task difficulty, sampling, or broken verifier | Inspect group standard deviations and known positive/negative cases |
| Rapid length growth | Length bias or stopping errors | Slice reward, accuracy, and truncation by length |
| KL spike | Large updates, stale samples, or reference mismatch | Check versions, learning rate, ratios, and reference |
| Sudden reward collapse | Unsynchronized weights or tokenizer mismatch | Check replica acknowledgments and token IDs |

Validate one synchronous rollout→update→synchronize cycle before optimizing asynchronous throughput. Record policy/reference versions, group size, sampling settings, reward version, KL coefficient, clip range, learning rate, effective batch, length cap, policy lag, and failed/timeout sample handling.

GRPO is not universally better than DPO. Start with DPO when reliable preference pairs exist; consider GRPO when verifiable rewards and online exploration justify it.

## 27. Capstone: Deliver a Reproducible Post-Training Experiment

Choose a bounded task such as structured extraction or verifiable arithmetic. These are acceptance requirements, not completed training results:

1. Write a one-page design: objective, primary metric, data provenance, split rules, budget, and stopping criteria.
2. Establish a base-model baseline. Overfit 8–32 examples, inspect token supervision, then use data with independent validation.
3. Run single-GPU SFT with complete checkpoints and a K→N recovery comparison. If resources permit, compare two GPUs at equal global batch.
4. Choose DPO or framework-supported synchronous GRPO. Change one primary factor and evaluate against SFT on the same sealed test set.
5. Serve explicitly verified weights/adapters with vLLM or SGLang. Check offline/serving differences and report latency, throughput, length, and correctness.
6. Deliver the manifest, configuration, per-example evaluation, raw logs, failure cases, and one page explaining confidence, bottlenecks, and how to falsify your interpretation.

Defense questions: Why might falling loss accompany worse accuracy? Why can two GPUs be slower? Why must EOS and PAD with the same ID be masked differently? Why is an adapter export insufficient for exact recovery? How do you handle serving/training log-prob discrepancies? Answering only "switch frameworks" is insufficient.

The new CPU correctness checks have been executed. GPU SFT, real LLM DPO/GRPO, recovery, and serving benchmarks remain untested here. This curriculum supplies a learning path and checkable components; execute the full capstone on your hardware.

Further reading: [PyTorch DDP](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html), [Transformers chat templates](https://huggingface.co/docs/transformers/chat_templating), [TRL DPO](https://huggingface.co/docs/trl/dpo_trainer), [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer), [Ray fault tolerance](https://docs.ray.io/en/latest/ray-core/fault_tolerance/index.html). These are reading references, not verification of every latest dependency combination.

In [ ]:
# Run manually after local experiments to shut down this process's Ray runtime:
# ray.shutdown()